In [2]:
!pip -q install pyspark

In [3]:
from google.colab import files
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from pyspark import StorageLevel
import time

uploaded = files.upload()

spark = SparkSession.builder \
    .appName("EnergyAssessment") \
    .getOrCreate()

def load_csv(filename):
    return spark.read.csv(filename, header=True, inferSchema=True)

readings_df = load_csv("meter_readings_90000.csv")
extra_readings_df = load_csv("meter_readings_extra_12000.csv")
meter_df = load_csv("meter_master.csv")
substation_df = load_csv("substation_master.csv")
tariff_df = load_csv("tariff_master.csv")
tickets_df = load_csv("maintenance_tickets_1600.csv")

print("All six datasets loaded successfully.")

Saving tariff_master.csv to tariff_master.csv
Saving substation_master.csv to substation_master.csv
Saving meter_readings_extra_12000.csv to meter_readings_extra_12000.csv
Saving meter_readings_90000.csv to meter_readings_90000.csv
Saving meter_master.csv to meter_master.csv
Saving maintenance_tickets_1600.csv to maintenance_tickets_1600.csv
All six datasets loaded successfully.


In [ ]:
readings_df.printSchema()

print("Total rows:", readings_df.count())

readings_df.show(10, truncate=False)

In [ ]:
selected_df = readings_df.select(
    "reading_id",
    "meter_id",
    "reading_timestamp",
    "energy_kwh",
    "reading_status"
)

selected_df.show(10)

renamed_df = readings_df.withColumnRenamed(
    "source_type", "collection_mode"
)

renamed_df.show(5)

In [ ]:
valid_high_energy_df = readings_df.filter(
    (F.col("reading_status") == "Valid") &
    (F.col("energy_kwh") > 20)
).orderBy(F.col("energy_kwh").desc())

valid_high_energy_df.show(20)

In [ ]:
energy_band_df = readings_df.withColumn(
    "energy_band",
    F.when(F.col("energy_kwh").isNull(), "Unknown")
     .when(F.col("energy_kwh") < 5, "Low")
     .when(F.col("energy_kwh") <= 25, "Medium")
     .otherwise("High")
)

energy_band_df.select(
    "reading_id", "energy_kwh", "energy_band"
).show(20)

In [ ]:
readings_df.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c + "_null_count")
    for c in [
        "energy_kwh",
        "voltage_v",
        "power_factor",
        "operator_id"
    ]
]).show()

In [ ]:
filled_df = readings_df.fillna({
    "operator_id": "UNASSIGNED",
    "power_factor": 0.90
})

filled_df.select(
    "reading_id", "operator_id", "power_factor"
).show(10)

In [ ]:
before_count = readings_df.count()

cleaned_df = readings_df.filter(
    F.col("energy_kwh").isNotNull() &
    F.col("voltage_v").isNotNull()
)

after_count = cleaned_df.count()

print("Before cleaning:", before_count)
print("After cleaning:", after_count)
print("Rows removed:", before_count - after_count)

In [ ]:
meter_readings_df = readings_df.join(
    meter_df,
    on="meter_id",
    how="inner"
)

site_energy_df = meter_readings_df.filter(
    F.col("reading_status") == "Valid"
).groupBy("site_type").agg(
    F.sum("energy_kwh").alias("total_energy_kwh")
)

site_energy_df.show()

In [ ]:
region_energy_df = meter_readings_df.join(
    substation_df,
    on="substation_id",
    how="inner"
).filter(
    F.col("reading_status") == "Valid"
).groupBy("region", "city").agg(
    F.sum("energy_kwh").alias("total_energy_kwh")
)

region_energy_df.show()

In [ ]:
tariff_readings_df = meter_readings_df.join(
    tariff_df,
    on="tariff_id",
    how="inner"
)

billable_df = tariff_readings_df.filter(
    (F.col("reading_status") != "Rejected") &
    F.col("energy_kwh").isNotNull()
).withColumn(
    "energy_charge_inr",
    F.col("energy_kwh") * F.col("rate_per_kwh_inr")
)

billable_df.select(
    "reading_id",
    "energy_kwh",
    "rate_per_kwh_inr",
    "energy_charge_inr"
).show(10)

In [ ]:
ticket_details_df = tickets_df.join(
    meter_df,
    on="meter_id",
    how="inner"
)

ticket_summary_df = ticket_details_df.groupBy(
    "site_type", "severity"
).agg(
    F.count("ticket_id").alias("ticket_count"),
    F.avg("maintenance_cost_inr").alias(
        "average_maintenance_cost_inr"
    )
)

ticket_summary_df.show()

In [ ]:
substation_summary_df = meter_readings_df.groupBy(
    "substation_id"
).agg(
    F.avg("voltage_v").alias("average_voltage"),
    F.sum("outage_minutes").alias("total_outage_minutes")
)

substation_summary_df.orderBy(
    F.col("total_outage_minutes").desc()
).show(5)

In [ ]:
source_summary_df = readings_df.filter(
    F.col("reading_status").isin(
        "Valid", "Estimated", "Rejected"
    )
).groupBy(
    "source_type"
).pivot(
    "reading_status",
    ["Valid", "Estimated", "Rejected"]
).count().fillna(0)

source_summary_df.show()

In [ ]:
timestamp_df = meter_readings_df.withColumn(
    "reading_timestamp",
    F.to_timestamp("reading_timestamp")
).withColumn(
    "year", F.year("reading_timestamp")
).withColumn(
    "month", F.month("reading_timestamp")
)

monthly_energy_df = timestamp_df.filter(
    F.col("reading_status") == "Valid"
).join(
    substation_df,
    on="substation_id",
    how="inner"
).groupBy(
    "region", "year", "month"
).agg(
    F.sum("energy_kwh").alias("monthly_energy_kwh")
)

monthly_energy_df.orderBy(
    "region", "year", "month"
).show()

In [ ]:
meter_energy_df = meter_readings_df.filter(
    F.col("reading_status") == "Valid"
).groupBy(
    "substation_id", "meter_id"
).agg(
    F.sum("energy_kwh").alias("total_energy_kwh")
)

rank_window = Window.partitionBy(
    "substation_id"
).orderBy(
    F.col("total_energy_kwh").desc()
)

ranked_meters_df = meter_energy_df.withColumn(
    "rank",
    F.dense_rank().over(rank_window)
)

ranked_meters_df.filter(
    F.col("rank") <= 3
).orderBy(
    "substation_id", "rank"
).show()

In [ ]:
time_window = Window.partitionBy(
    "meter_id"
).orderBy(
    "reading_timestamp", "reading_id"
).rowsBetween(
    Window.unboundedPreceding,
    Window.currentRow
)

cumulative_df = readings_df.filter(
    F.col("reading_status") == "Valid"
).withColumn(
    "reading_timestamp",
    F.to_timestamp("reading_timestamp")
).withColumn(
    "cumulative_energy_kwh",
    F.sum("energy_kwh").over(time_window)
)

cumulative_df.select(
    "meter_id",
    "reading_timestamp",
    "reading_id",
    "energy_kwh",
    "cumulative_energy_kwh"
).show(20)

In [ ]:
print("Main readings:", readings_df.count())
print("Extra readings:", extra_readings_df.count())

combined_df = readings_df.unionByName(extra_readings_df)

print("Combined readings:", combined_df.count())

combined_df.show(5)

In [ ]:
duplicate_ids_df = combined_df.groupBy(
    "reading_id"
).count().filter(
    F.col("count") > 1
)

print("Repeated reading IDs:")
duplicate_ids_df.show()

unique_readings_df = combined_df.dropDuplicates(
    ["reading_id"]
)

print("Rows before removing duplicates:", combined_df.count())
print("Final unique readings:", unique_readings_df.count())

In [ ]:
pipeline_df = readings_df.filter(
    F.col("reading_status") == "Valid"
).withColumn(
    "energy_band",
    F.when(F.col("energy_kwh") < 5, "Low")
     .when(F.col("energy_kwh") <= 25, "Medium")
     .otherwise("High")
).groupBy(
    "energy_band"
).agg(
    F.sum("energy_kwh").alias("total_energy")
).orderBy(
    F.col("total_energy").desc()
)

pipeline_df.explain()
pipeline_df.show()

In [ ]:
valid_df = readings_df.filter(
    F.col("reading_status") == "Valid"
).cache()

start = time.time()
count1 = valid_df.count()
time1 = time.time() - start

start = time.time()
count2 = valid_df.count()
time2 = time.time() - start

print("First count:", count1)
print("First execution time:", time1)

print("Second count:", count2)
print("Second execution time:", time2)

print("Is cached:", valid_df.is_cached)

valid_df.unpersist()

In [ ]:
outage_df = readings_df.filter(
    F.col("outage_minutes") > 0
).persist(StorageLevel.MEMORY_AND_DISK)

print("Outage rows:", outage_df.count())

outage_df.groupBy("reading_status").count().show()

outage_df.unpersist()
print("Persisted data released.")

In [ ]:
start = time.time()

readings_df.filter(
    F.col("reading_status") == "Valid"
).groupBy(
    "meter_id"
).agg(
    F.sum("energy_kwh").alias("total_energy")
).count()

without_cache_time = time.time() - start
valid_cached_df = readings_df.filter(
    F.col("reading_status") == "Valid"
).cache()
valid_cached_df.count()

start = time.time()

valid_cached_df.groupBy(
    "meter_id"
).agg(
    F.sum("energy_kwh").alias("total_energy")
).count()

with_cache_time = time.time() - start


print("Without cache:", without_cache_time)
print("With cache:", with_cache_time)

valid_cached_df.unpersist()

In [ ]:
eight_partition_df = readings_df.repartition(8)

partition_counts_df = eight_partition_df.withColumn(
    "partition_id",
    F.spark_partition_id()
).groupBy(
    "partition_id"
).count().orderBy(
    "partition_id"
)

partition_counts_df.show()

In [ ]:
twelve_partition_df = readings_df.repartition(12)

coalesce_df = twelve_partition_df.coalesce(3)
repartition_df = twelve_partition_df.repartition(3)

print("Original partitions:", twelve_partition_df.rdd.getNumPartitions())
print("Coalesce partitions:", coalesce_df.rdd.getNumPartitions())
print("Repartition partitions:", repartition_df.rdd.getNumPartitions())

print("Coalesce execution plan:")
coalesce_df.explain()

print("Repartition execution plan:")
repartition_df.explain()

In [ ]:
enriched_df = cleaned_df.join(
    meter_df,
    on="meter_id",
    how="inner"
).join(
    substation_df,
    on="substation_id",
    how="inner"
)

output_df = enriched_df.repartition(4)

output_df.write.mode("overwrite").parquet(
    "energy_enriched_parquet"
)

print("Parquet file saved successfully.")

import os

print(os.listdir("energy_enriched_parquet"))

***Capstone Project***

In [ ]:
capstone_readings_df = readings_df.unionByName(
    extra_readings_df
).dropDuplicates(["reading_id"])

print("Unique readings:", capstone_readings_df.count())

capstone_clean_df = capstone_readings_df.filter(
    F.col("energy_kwh").isNotNull() &
    (F.col("reading_status") != "Rejected")
)

capstone_clean_df = capstone_clean_df.withColumn(
    "reading_timestamp",
    F.to_timestamp("reading_timestamp")
)

capstone_joined_df = capstone_clean_df.join(
    meter_df,
    on="meter_id",
    how="inner"
).join(
    substation_df,
    on="substation_id",
    how="inner"
).join(
    tariff_df,
    on="tariff_id",
    how="inner"
)

capstone_joined_df = capstone_joined_df.withColumn(
    "month",
    F.date_format("reading_timestamp", "yyyy-MM")
).withColumn(
    "energy_charge_inr",
    F.col("energy_kwh") * F.col("rate_per_kwh_inr")
)

capstone_report_df = capstone_joined_df.groupBy(
    "region", "site_type", "month"
).agg(
    F.sum("energy_kwh").alias("total_kwh"),

    F.sum("energy_charge_inr").alias(
        "total_energy_charges_inr"
    ),

    F.sum(
        F.coalesce(F.col("outage_minutes"), F.lit(0))
    ).alias("total_outage_minutes"),

    F.count("*").alias("total_readings"),

    F.sum(
        F.when(F.col("outage_minutes") > 0, 1)
         .otherwise(0)
    ).alias("outage_readings")
).withColumn(
    "outage_percentage",
    F.round(
        F.col("outage_readings") /
        F.col("total_readings") * 100,
        2
    )
)

capstone_window = Window.partitionBy(
    "region", "month"
).orderBy(
    F.col("total_energy_charges_inr").desc()
)

capstone_report_df = capstone_report_df.withColumn(
    "site_type_rank",
    F.dense_rank().over(capstone_window)
)

capstone_report_df = capstone_report_df.orderBy(
    "region", "month", "site_type_rank"
)

capstone_report_df.show(100, truncate=False)

capstone_report_df.cache()

print("First action - row count:", capstone_report_df.count())

print("Second action - report preview:")
capstone_report_df.show(20, truncate=False)

print("Is report cached:", capstone_report_df.is_cached)

capstone_report_df.unpersist()

capstone_report_df.repartition(2).write.mode(
    "overwrite"
).parquet("energy_monthly_report_parquet")

print("Final monthly report saved successfully.")

print(os.listdir("energy_monthly_report_parquet"))